In [ ]:
pip install --upgrade torch ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━ 434.6/664.8 MB 226.3 MB/s eta 0:00:02

In [ ]:
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.engine.model import Model
import os, pandas, numpy, cv2
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn

In [ ]:
CLASSES=['labels']

yaml_content = f"""
train: /kaggle/input/simulated-object-data/data/train/images
val: /kaggle/input/simulated-object-data/data/val/images

nc: {len(CLASSES)}
names: {CLASSES}
"""
    
with open("dataset.yaml", "w") as f:
    f.write(yaml_content)

print("dataset.yaml created!")

In [ ]:
#model.add_module('custom_head', torch.nn.Sequential())

In [ ]:
class Conv(nn.Module):
    default_act = nn.ReLU()#inplace=True)

    def __init__(self, c1, c2, kernel_size=1, stride=1, padding=None, act=True, eps = 0.001, momentum = 0.03):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(c2, eps, momentum)
        self.act = self.default_act if act is True else act if isinstance(act, nn.Module) else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))
        
class RE_Conv(nn.Module):
    default_act = nn.ReLU()#inplace=True)

    def __init__(self, c1, c2, kernel_size=1, stride=1, padding=None, act=True, eps = 0.001, momentum = 0.03, groups = 1):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, kernel_size, stride, padding, groups, bias=False)
        self.bn = nn.BatchNorm2d(c2, eps, momentum)
        self.act = self.default_act if act is True else act if isinstance(act, nn.Module) else nn.Identity()

    def forward(self, x):
        return self.act(self.conv(x))

In [ ]:
class C3k2(nn.Module):
    def __init__(self, x_0, x_1, x_2, x_3, x_4):
        super().__init__()

        self.cv1 = Conv(x_0, x_1, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
        self.cv2 = Conv(x_2, x_3, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))

        self.m = nn.ModuleList([
                      Conv(x_4//2, x_4//4, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1)),
                      Conv(x_4//2, x_4//4, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1)),
                      Conv(x_4//2, x_4//2, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1)),
                
                      nn.Sequential(
                          Conv(x_4//4, x_4//4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                          Conv(x_4//4, x_4//4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                      
                          Conv(x_4//4, x_4//4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                          Conv(x_4//4, x_4//4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
                      )]
                  )
    def forward(self, x):
        x = self.cv1(x)
        x = self.cv2(x)
        return self.m(x)

In [ ]:
class Detect(nn.Module):
    def __init__(self):
        super().__init__()

        self.cv1 = nn.ModuleList([
                        nn.Sequential(
                            Conv(384, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                            Conv(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                            nn.Conv2d(96, 64, kernel_size=(1, 1), stride=(1, 1))
                        ),
                        nn.Sequential(
                            Conv(768, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                            Conv(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                            nn.Conv2d(96, 64, kernel_size=(1, 1), stride=(1, 1))
                        )
                   ])
        self.cv2 = nn.ModuleList([
                        nn.Sequential(
                            nn.Sequential(
                                RE_Conv(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=384),
                                Conv(384, 384, kernel_size=(1, 1), stride=(1, 1))
                            ),
                            nn.Sequential(
                               RE_Conv(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=384),
                               Conv(384, 384, kernel_size=(1, 1), stride=(1, 1))
                           ),
                          nn.Conv2d(384, 1, kernel_size=(1, 1), stride=(1, 1))
                       ),
                       nn.Sequential(
                           nn.Sequential(
                                RE_Conv(768, 768, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=768),
                                Conv(768, 384, kernel_size=(1, 1), stride=(1, 1))
                          ),
                          nn.Sequential(
                                RE_Conv(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=384),
                                Conv(384, 384, kernel_size=(1, 1), stride=(1, 1))
                          ),
                          nn.Conv2d(384, 1, kernel_size=(1, 1), stride=(1, 1)),
                       )
                   ])

        self.m = nn.Conv2d(1, 1, kernel_size=(1, 1), stride=(1, 1), bias=False)
        
    def forward(self, x):
        x = self.cv1(x),
        x = self.cv2(x),
        return self.m(x)

In [ ]:
yolo = YOLO("yolo11x.pt")

In [ ]:
yolo.model.model[0] = Conv(3, 296, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[1] = Conv(296, 192, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[2] = C3k2(192, 192, 384, 384, 192)
yolo.model.model[3] = Conv(384, 384, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[4] = C3k2(384, 384, 768, 768, 384)
yolo.model.model[5] = Conv(768, 768, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[6] = C3k2(768, 768, 1536, 768, 768)
yolo.model.model[7] = Conv(768, 768, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[8] = C3k2(768, 768, 1536, 768, 768)
yolo.model.model[13] = C3k2(1536, 768, 1536, 768, 768)
yolo.model.model[16] = C3k2(1536, 384, 768, 384, 384)
yolo.model.model[17] = Conv(384, 384, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[19] = C3k2(1152, 768, 1152, 768, 768)
yolo.model.model[20] = Conv(768, 768, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[22] = C3k2(1536, 768, 1536, 768, 768)
yolo.model.model[23] = Detect()

In [ ]:
yolo.save("model_11x.pt")

In [ ]:
model_yaml = f"""
nc: 80 # number of classes
scales: # model compound scaling constants, i.e. 'model=yolo11n.yaml' will call yolo11.yaml with scale 'n'
  # [depth, width, max_channels]
  x: [1.00, 1.50, 512] # summary: 357 layers, 56966176 parameters, 56966160 gradients, 196.0 GFLOPs

# YOLO11n backbone
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 1]] # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

# YOLO11n head
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, False]] # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, False]] # 16 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, False]] # 19 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 2, C3k2, [1024, True]] # 22 (P5/32-large)

  - [[16, 19, 22], 1, Detect, [nc]] # Detect(P3, P4, P5)
"""

with open("model.yaml", "w") as f:
    f.write(model_yaml)

In [ ]:
from ultralytics import RTDETR

model = YOLO("model.yaml")

In [ ]:
model.train( data='/kaggle/working/dataset.yaml',
            epochs=150,
            batch=10,
            imgsz=724,
            patience=5,
            lr0=0.001,
            lrf=0.02,
            optimizer="SGD",
            momentum=0.96,
            weight_decay=0.001,
            cos_lr=True,
            dropout=0.3,
            label_smoothing=0.1,
            mosaic=0.5,
            mixup=0.15,
            copy_paste=0.1,
            fliplr=0.5,
            flipud=0.5,
            hsv_h=0.5,
            hsv_s=0.9,
            hsv_v=0.9,
            translate=0.2,
            scale=0.5,
            shear=0.2,
            perspective=0.0002,
            val=True,
            workers=8,
            seed=42,
            device=[-1, -1]
        )
valid_results = yolo.val()
print(valid_results)

In [ ]:
model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

In [ ]:
output_dir = r"/kaggle/working/predictions/labels"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
for i in os.listdir('/kaggle/input/multi-instance-object-detection-challenge/Starter_Dataset/TestImages/images'):
    img_path = f'/kaggle/input/multi-instance-object-detection-challenge/Starter_Dataset/TestImages/images/{i}'
    results = model.predict(img_path, 
                            conf=0.3, device=0, verbose=False) # 0 - GPU or "cpu" Image.fromarray(cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2HSV))
    output_txt = f"{output_dir}/{i.split('.')[0]}.txt"

    with open(output_txt, "w") as f:
        found = False
        for result in results:
            img_height, img_width = result.orig_shape
            boxes = result.boxes.data

            if boxes is None or len(boxes) == 0:
                continue

            filtered_boxes = boxes[boxes[:, 4] >= 0.05]
            if len(filtered_boxes) == 0:
                continue

            found = True
            for box in filtered_boxes:
                x1, y1, x2, y2, confidence, cls_id = box.tolist()

                x_center = ((x1 + x2) / 2) / img_width
                y_center = ((y1 + y2) / 2) / img_height
                width = (x2 - x1) / img_width
                height = (y2 - y1) / img_height

                f.write(f"0 {confidence:.6f} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

        if not found:
            f.write("")

In [ ]:
rows = []
output_dir = Path("/kaggle/working/predictions/labels")
TEST = Path('/kaggle/input/multi-instance-object-detection-challenge/Starter_Dataset/TestImages/images')
test_imgs = {p.stem for p in TEST.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
predicted = set()

for file in output_dir.glob("*.txt"):
    name = file.stem
    predicted.add(name)

    try:
        lines = [l.strip() for l in open(file) if len(l.strip().split()) == 6]
    except:
        lines = []

    rows.append({"image_id": name, "prediction_string": " ".join(lines) if lines else "no boxes"})

for name in test_imgs - predicted:
    rows.append({"image_id": name, "prediction_string": "no boxes"})

work_dir = '/kaggle/working'

for filename in os.listdir(work_dir):
    file_path = os.path.join(work_dir, filename)

    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print(f'Error file: {file_path}. Cause: {e}')

rows = pandas.DataFrame(rows)
rows.to_csv("submission.csv", index=False)
rows